# RoadSafe Analytics
## 03 — Data Integration & Feature Engineering

### Objective

Integrate the cleaned STATS19 collision, vehicle, and casualty datasets into structured analytical datasets suitable for exploratory analysis and machine learning.

### Key Tasks

1. Examine relationships between the three datasets.
2. Aggregate vehicle and casualty information.
3. Integrate datasets while preserving one row per collision.
4. Engineer temporal, geographical, and road-related features.
5. Prepare separate classification and regression datasets.
6. Prevent target leakage.
7. Validate and export the integrated datasets.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Project directories
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
interim_dir = project_root / "data" / "interim"

print("Project root:", project_root)
print("Processed data directory:", processed_dir)

Project root: c:\Users\Tanishka Singh\OneDrive\Desktop\UPES\Semester-3\FDS\Project\RoadSafe-Analytics
Processed data directory: c:\Users\Tanishka Singh\OneDrive\Desktop\UPES\Semester-3\FDS\Project\RoadSafe-Analytics\data\processed


In [2]:
collisions = pd.read_parquet(processed_dir / "collisions_clean.parquet")
vehicles = pd.read_parquet(processed_dir / "vehicles_clean.parquet")
casualties = pd.read_parquet(processed_dir / "casualties_clean.parquet")

print("Collisions:", collisions.shape)
print("Vehicles:", vehicles.shape)
print("Casualties:", casualties.shape)

Collisions: (101525, 43)
Vehicles: (183948, 31)
Casualties: (127883, 24)


In [3]:
for name, df in {
    "Collisions": collisions,
    "Vehicles": vehicles,
    "Casualties": casualties
}.items():

    identifier_columns = [col for col in df.columns if "index" in col or "reference" in col]

    print(f"\n{name}:")
    print(identifier_columns)


Collisions:
['collision_index']

Vehicles:
['collision_index', 'vehicle_reference']

Casualties:
['collision_index', 'vehicle_reference', 'casualty_reference']


## Understanding Dataset Relationships

The STATS19 datasets are related through collision identifiers.

- Each collision has a unique `collision_index`.
- Multiple vehicles can be associated with one collision.
- Multiple casualties can be associated with one collision.
- Vehicle and casualty records contain additional reference identifiers.

Directly joining the three datasets may duplicate collision records.

Therefore, related information will be aggregated to the collision level before integration, preserving one row per collision.

In [4]:
relationship_summary = pd.DataFrame({
    "Dataset": ["Collisions", "Vehicles", "Casualties"],
    "Total Records": [len(collisions),len(vehicles),len(casualties)],
    "Unique Collisions": [collisions["collision_index"].nunique(),vehicles["collision_index"].nunique(),casualties["collision_index"].nunique()]
    })

display(relationship_summary)

,Dataset,Total Records,Unique Collisions
0,Collisions,101525,101525
1,Vehicles,183948,101525
2,Casualties,127883,101525


## Vehicle Data Aggregation

The vehicle dataset contains multiple records for some collisions.

To avoid duplicating collision records during integration, vehicle information is aggregated using `collision_index`.

We first calculate the number of vehicle records associated with each collision and compare it with the recorded vehicle count in the collision dataset.

This also provides an additional consistency check before feature engineering.

In [5]:
vehicle_counts = (vehicles.groupby("collision_index").size().reset_index(name="vehicle_record_count"))

print("Aggregated vehicle data shape:", vehicle_counts.shape)

display(vehicle_counts.head())

Aggregated vehicle data shape: (101525, 2)


,collision_index,vehicle_record_count
0,2025010551784,3
1,2025010551786,2
2,2025010551792,5
3,2025010551794,1
4,2025010551795,2


In [6]:
vehicle_count_check = collisions[["collision_index", "number_of_vehicles"]].merge(
    vehicle_counts,
    on="collision_index",
    how="left",
    validate="one_to_one"
)

mismatches = (vehicle_count_check["number_of_vehicles"]!= vehicle_count_check["vehicle_record_count"])

print("Vehicle count mismatches:", mismatches.sum())

display(vehicle_count_check.head())

Vehicle count mismatches: 0


,collision_index,number_of_vehicles,vehicle_record_count
0,202517M101925,1,1
1,202517L205225,2,2
2,2025070971700,2,2
3,2025070403274,2,2
4,2025071000604,7,7


### Inspection of Vehicle Attributes

The vehicle dataset contains information about vehicles and drivers involved in reported collisions.

Before aggregating additional attributes, we inspect the available columns and their data types to select meaningful features for collision-level analysis.

In [7]:
vehicle_columns = pd.DataFrame({
    "Column": vehicles.columns,
    "Data Type": vehicles.dtypes.astype(str).values,
    "Missing Values": vehicles.isna().sum().values,
    "Missing (%)": (vehicles.isna().mean().values * 100).round(2)
    })

display(vehicle_columns)

,Column,Data Type,Missing Values,Missing (%)
0,collision_index,string,0,0.00
1,collision_year,int64,0,0.00
2,collision_ref_no,string,0,0.00
3,vehicle_reference,int64,0,0.00
4,vehicle_type,float64,9,0.00
5,towing_and_articulation,float64,1865,1.01
6,vehicle_manoeuvre_historic,float64,161547,87.82
7,vehicle_manoeuvre,float64,5198,2.83
8,vehicle_direction_from,float64,6832,3.71
9,vehicle_direction_to,float64,6865,3.73


### Collision-Level Vehicle Feature Engineering

Vehicle records are aggregated using `collision_index` to create summary features for each collision.

The following features are calculated:

- Mean age of drivers with known ages.
- Number of drivers with known ages.
- Number of distinct recorded vehicle types.

Missing driver ages are excluded from the mean rather than replaced with arbitrary values.

The original vehicle dataset remains unchanged.

In [8]:
vehicle_features = (vehicles.groupby("collision_index").agg(
        mean_driver_age=("age_of_driver", "mean"),
        known_driver_age_count=("age_of_driver", "count"),
        distinct_vehicle_types=("vehicle_type", "nunique")
        ).reset_index())

print("Vehicle features shape:", vehicle_features.shape)

display(vehicle_features.head())

Vehicle features shape: (101525, 4)


,collision_index,mean_driver_age,known_driver_age_count,distinct_vehicle_types
0,2025010551784,48.0,3,1
1,2025010551786,39.5,2,2
2,2025010551792,29.0,1,1
3,2025010551794,40.0,1,1
4,2025010551795,43.5,2,2


## Casualty Data Aggregation

The casualty dataset contains individual casualty records associated with reported collisions.

We aggregate casualty records by `collision_index` to obtain the number of casualties recorded for each collision.

The aggregated counts are compared with the collision dataset to verify consistency.

Casualty-derived information will be retained for descriptive analysis but excluded from the initial predictive feature sets to prevent target leakage.

In [9]:
casualty_counts = (casualties.groupby("collision_index").size().reset_index(name="casualty_record_count"))

print("Aggregated casualty data shape:", casualty_counts.shape)

display(casualty_counts.head())

Aggregated casualty data shape: (101525, 2)


,collision_index,casualty_record_count
0,2025010551784,1
1,2025010551786,1
2,2025010551792,1
3,2025010551794,2
4,2025010551795,1


In [10]:
casualty_count_check = collisions[["collision_index", "number_of_casualties"]].merge(
    casualty_counts,
    on="collision_index",
    how="left",
    validate="one_to_one"
)

mismatches = (casualty_count_check["number_of_casualties"]!= casualty_count_check["casualty_record_count"])

print("Casualty count mismatches:", mismatches.sum())

display(casualty_count_check.head())

Casualty count mismatches: 0


,collision_index,number_of_casualties,casualty_record_count
0,202517M101925,1,1
1,202517L205225,1,1
2,2025070971700,2,2
3,2025070403274,1,1
4,2025071000604,1,1


### Collision-Level Casualty Feature Engineering

The casualty dataset contains individual casualty records associated with each collision.

To support descriptive analysis, we aggregate casualty information using `collision_index`.

The following features are created:

1. Mean age of casualties with known ages.
2. Number of casualties with known ages.
3. Number of distinct casualty types.

These features are intended for descriptive analysis only and will be excluded from predictive models to prevent target leakage.

The original cleaned datasets remain unchanged.

In [11]:
casualty_features = (casualties.groupby("collision_index").agg(
        mean_casualty_age=("age_of_casualty", "mean"),
        known_casualty_age_count=("age_of_casualty", "count"),
        distinct_casualty_types=("casualty_type", "nunique")
        ).reset_index())

print("Casualty features shape:", casualty_features.shape)

display(casualty_features.head())

Casualty features shape: (101525, 4)


,collision_index,mean_casualty_age,known_casualty_age_count,distinct_casualty_types
0,2025010551784,55.0,1,1
1,2025010551786,30.0,1,1
2,2025010551792,29.0,1,1
3,2025010551794,59.0,1,1
4,2025010551795,25.0,1,1


## Integration of Collision, Vehicle, and Casualty Data

The cleaned collision dataset is combined with the aggregated vehicle and casualty features using `collision_index`.

Left joins are used to preserve all collision records.

The integration produces one row per collision and prevents the duplication that would occur when directly joining individual vehicle and casualty records.

Casualty-derived features are retained for descriptive analysis and will be excluded from predictive modelling inputs.

In [12]:
integrated_df = (collisions.merge(vehicle_features,on="collision_index",how="left",validate="one_to_one").merge(
        casualty_features,
        on="collision_index",
        how="left",
        validate="one_to_one"
        ))

print("Integrated dataset shape:", integrated_df.shape)

display(integrated_df.head())

Integrated dataset shape: (101525, 49)


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,mean_driver_age,known_driver_age_count,distinct_vehicle_types,mean_casualty_age,known_casualty_age_count,distinct_casualty_types
0,202517M101925,2025,17M101925,447673.0,519279.0,-1.26420,54.56650,17,3,1,...,3.0,1,0.0,1.0,NaN,0,1,11.0,1,1
1,202517L205225,2025,17L205225,454611.0,518597.0,-1.15704,54.55967,17,3,2,...,3.0,1,0.0,1.0,44.0,2,1,52.0,1,1
2,2025070971700,2025,070971700,354476.0,382656.0,-2.68516,53.33884,7,3,2,...,3.0,1,0.0,1.0,35.5,2,1,35.5,2,1
3,2025070403274,2025,070403274,361793.0,393163.0,-2.57655,53.43386,7,3,2,...,3.0,1,0.0,1.0,35.5,2,2,31.0,1,1
4,2025071000604,2025,071000604,366083.0,389327.0,-2.51157,53.39967,7,3,7,...,3.0,1,0.0,1.0,41.0,7,2,38.0,1,1


In [13]:
integration_checks = {
    "Row count preserved": len(integrated_df) == len(collisions),
    "Collision IDs unique": integrated_df["collision_index"].is_unique,
    "No missing vehicle summaries": integrated_df["known_driver_age_count"].notna().all(),
    "No missing casualty summaries": integrated_df["known_casualty_age_count"].notna().all(),
    "Collision IDs preserved": integrated_df["collision_index"].equals(collisions["collision_index"])
    }

for check, passed in integration_checks.items():
    print(f"{check}: {passed}")

print("\nAll integration checks passed:", all(integration_checks.values()))

Row count preserved: True
Collision IDs unique: True
No missing vehicle summaries: True
No missing casualty summaries: True
Collision IDs preserved: True

All integration checks passed: True


## Temporal Feature Engineering

Temporal features are extracted from collision dates and times to identify patterns in road accidents.

The following features are created:

1. Month of collision.
2. Day of the week.
3. Hour of collision.
4. Weekend indicator.

These features will support exploratory data analysis and predictive modelling.

In [17]:
# Create temporal features from collision date and time

integrated_df["date"] = pd.to_datetime(integrated_df["date"])
integrated_df["month"] = integrated_df["date"].dt.month
integrated_df["day_of_week"] = integrated_df["date"].dt.isocalendar().day - 1
integrated_df["hour"] = pd.to_datetime(integrated_df["time"],format="%H:%M", errors="coerce").dt.hour
integrated_df["is_weekend"] = (integrated_df["day_of_week"] >= 5).astype(int)

display(integrated_df[["date", "time", "month", "day_of_week", "hour", "is_weekend"]].head())

,date,time,month,day_of_week,hour,is_weekend
0,2025-02-04,15:00,2,1,15,0
1,2025-05-07,09:35,5,2,9,0
2,2025-11-27,18:18,11,3,18,0
3,2025-05-15,16:30,5,3,16,0
4,2025-12-08,07:45,12,0,7,0


## Additional Feature Engineering

Additional features are created to make collision patterns easier to interpret.

The first feature categorizes collisions into four time periods:

- Night: 00:00–05:59
- Morning: 06:00–11:59
- Afternoon: 12:00–17:59
- Evening: 18:00–23:59

This feature will support time-based exploratory analysis.

In [18]:
integrated_df["time_of_day"] = pd.cut(integrated_df["hour"],bins=[-1, 5, 11, 17, 23],labels=["Night", "Morning", "Afternoon", "Evening"])

display(integrated_df[["time", "hour", "time_of_day"]].head(10))

print("\nTime-of-day distribution:")
print(integrated_df["time_of_day"].value_counts())

,time,hour,time_of_day
0,15:00,15,Afternoon
1,09:35,9,Morning
2,18:18,18,Evening
3,16:30,16,Afternoon
4,07:45,7,Morning
5,12:11,12,Afternoon
6,11:45,11,Morning
7,17:31,17,Afternoon
8,16:00,16,Afternoon
9,16:40,16,Afternoon



Time-of-day distribution:
time_of_day
Afternoon    44365
Morning      26843
Evening      24734
Night         5583
Name: count, dtype: int64


## Preparing Machine Learning Datasets

Separate datasets are prepared for classification and regression.

Classification Target:
- collision_severity

Regression Target:
- number_of_casualties

Casualty-derived features are excluded from both predictive datasets to prevent target leakage.

The datasets will contain collision-level information and selected engineered features.

Final preprocessing, encoding, imputation, and train-test splitting will be performed during the modelling stages.

In [19]:
print("Integrated dataset shape:", integrated_df.shape)

print("\nAvailable columns:")
for column in integrated_df.columns:
    print(column)

Integrated dataset shape: (101525, 53)

Available columns:
collision_index
collision_year
collision_ref_no
location_easting_osgr
location_northing_osgr
longitude
latitude
police_force
collision_severity
number_of_vehicles
number_of_casualties
date
day_of_week
time
local_authority_ons_district
local_authority_highway
local_authority_highway_current
first_road_class
first_road_number
road_type
speed_limit
junction_detail_historic
junction_detail
junction_control
second_road_class
second_road_number
pedestrian_crossing_human_control_historic
pedestrian_crossing_physical_facilities_historic
pedestrian_crossing
light_conditions
weather_conditions
road_surface_conditions
special_conditions_at_site
carriageway_hazards_historic
carriageway_hazards
urban_or_rural_area
did_police_officer_attend_scene_of_accident
trunk_road_flag
lsoa_of_accident_location
enhanced_severity_collision
collision_injury_based
collision_adjusted_severity_serious
collision_adjusted_severity_slight
mean_driver_age
known_

### Selection of Predictive Features

Separate feature sets are prepared for classification and regression.

The selected predictors describe road characteristics, environmental conditions, and collision timing.

Collision identifiers, casualty-derived attributes, and outcome variables are excluded to reduce target leakage.

The resulting models will estimate outcomes for recorded collisions, not predict whether a collision will occur.

Missing-value imputation and categorical encoding will be performed after train-test splitting during the modelling stages.

In [22]:
# Initial predictors shared by both models
candidate_features = [
    "number_of_vehicles",
    "speed_limit",
    "road_type",
    "first_road_class",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "month",
    "day_of_week",
    "hour",
    "is_weekend"
]

# Check which features are available
available_features = [col for col in candidate_features if col in integrated_df.columns]

missing_features = [col for col in candidate_features if col not in integrated_df.columns]

print("Available features:", available_features)
print("\nMissing features:", missing_features)

Available features: ['number_of_vehicles', 'speed_limit', 'road_type', 'first_road_class', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'month', 'day_of_week', 'hour', 'is_weekend']

Missing features: []


### Creation of Classification and Regression Datasets

Two separate machine-learning datasets are prepared using the selected collision-level predictors.

Classification target: collision_severity
Regression target: number_of_casualties

Each dataset contains 11 predictor columns and one target column.

Casualty-derived features, collision identifiers, and the other model's target are excluded to prevent target leakage.

Feature encoding, missing-value imputation, and train-test splitting will be performed during the modelling stages.

In [23]:
classification_df = integrated_df[available_features + ["collision_severity"]].copy()

regression_df = integrated_df[available_features + ["number_of_casualties"]].copy()

print("Classification dataset:", classification_df.shape)
print("Regression dataset:", regression_df.shape)

print("\nClassification sample:")
display(classification_df.head())

print("\nRegression sample:")
display(regression_df.head())

Classification dataset: (101525, 12)
Regression dataset: (101525, 12)

Classification sample:


,number_of_vehicles,speed_limit,road_type,first_road_class,light_conditions,weather_conditions,road_surface_conditions,month,day_of_week,hour,is_weekend,collision_severity
0,1,30,6.0,4.0,1.0,1.0,1.0,2,1,15,0,3
1,2,30,6.0,5.0,1.0,1.0,1.0,5,2,9,0,3
2,2,30,6.0,6.0,4.0,1.0,2.0,11,3,18,0,3
3,2,40,6.0,5.0,1.0,1.0,1.0,5,3,16,0,3
4,7,70,3.0,1.0,4.0,1.0,2.0,12,0,7,0,3



Regression sample:


,number_of_vehicles,speed_limit,road_type,first_road_class,light_conditions,weather_conditions,road_surface_conditions,month,day_of_week,hour,is_weekend,number_of_casualties
0,1,30,6.0,4.0,1.0,1.0,1.0,2,1,15,0,1
1,2,30,6.0,5.0,1.0,1.0,1.0,5,2,9,0,1
2,2,30,6.0,6.0,4.0,1.0,2.0,11,3,18,0,2
3,2,40,6.0,5.0,1.0,1.0,1.0,5,3,16,0,1
4,7,70,3.0,1.0,4.0,1.0,2.0,12,0,7,0,1


## Final Machine Learning Dataset Validation

The classification and regression datasets are validated before export.

Validation checks include:

1. Preservation of the original collision count.
2. Correct number of predictors.
3. Absence of target leakage between datasets.
4. Presence of valid target values.
5. Preservation of the original target values.
6. Identification of missing predictor values.

Missing predictor values will be handled during model training using appropriate preprocessing pipelines.

In [24]:
validation_checks = {"Classification row count preserved":len(classification_df) == len(collisions),
                     "Regression row count preserved": len(regression_df) == len(collisions),
                     "Classification has 11 predictors":classification_df.shape[1] == 12,
                     "Regression has 11 predictors": regression_df.shape[1] == 12,
                     "Casualty count excluded from classification": "number_of_casualties" not in classification_df.columns,
                     "Collision severity excluded from regression":"collision_severity" not in regression_df.columns,
                     "Classification target has no missing values":classification_df["collision_severity"].notna().all(),
                     "Regression target has no missing values":regression_df["number_of_casualties"].notna().all(),
                     "Classification target preserved": classification_df["collision_severity"].equals(collisions["collision_severity"]),
                     "Regression target preserved":regression_df["number_of_casualties"].equals(collisions["number_of_casualties"])
                     }

validation_df = pd.DataFrame({"Validation Check": validation_checks.keys(),"Passed": validation_checks.values()})

display(validation_df)

print("All validation checks passed:",all(validation_checks.values()))

,Validation Check,Passed
0,Classification row count preserved,True
1,Regression row count preserved,True
2,Classification has 11 predictors,True
3,Regression has 11 predictors,True
4,Casualty count excluded from classification,True
5,Collision severity excluded from regression,True
6,Classification target has no missing values,True
7,Regression target has no missing values,True
8,Classification target preserved,True
9,Regression target preserved,True


All validation checks passed: True


In [25]:
print("Classification missing values:")
display(classification_df[available_features].isna().sum().sort_values(ascending=False))

print("\nRegression missing values:")
display(regression_df[available_features].isna().sum().sort_values(ascending=False))

Classification missing values:


weather_conditions         2385
road_type                  1953
road_surface_conditions     634
light_conditions             16
first_road_class              6
speed_limit                   0
number_of_vehicles            0
month                         0
day_of_week                   0
hour                          0
is_weekend                    0
dtype: int64


Regression missing values:


weather_conditions         2385
road_type                  1953
road_surface_conditions     634
light_conditions             16
first_road_class              6
speed_limit                   0
number_of_vehicles            0
month                         0
day_of_week                   0
hour                          0
is_weekend                    0
dtype: int64

## Export of Final Stage 5 Datasets

Three datasets are exported in Parquet format:

1. Integrated analytical dataset for exploratory analysis.
2. Classification dataset for collision severity prediction.
3. Regression dataset for casualty count prediction.

The predictive datasets contain 11 selected features and their respective target variables.

Missing-value imputation, categorical encoding, and train-test splitting will be performed during model development.

Original cleaned datasets remain unchanged.

In [26]:
# Create output directory
processed_dir.mkdir(parents=True, exist_ok=True)

# Export integrated analytical dataset
integrated_df.to_parquet(processed_dir / "integrated_analytical.parquet",index=False)

# Export classification dataset
classification_df.to_parquet(processed_dir / "classification_ready.parquet",index=False)

# Export regression dataset
regression_df.to_parquet(processed_dir / "regression_ready.parquet",index=False)

print("All three Stage 5 datasets exported successfully!")

All three Stage 5 datasets exported successfully!


In [27]:
exported_datasets = {"Integrated": "integrated_analytical.parquet","Classification": "classification_ready.parquet","Regression": "regression_ready.parquet"}

original_datasets = {"Integrated": integrated_df,"Classification": classification_df,"Regression": regression_df}

for name, filename in exported_datasets.items():

    path = processed_dir / filename
    saved_df = pd.read_parquet(path)

    print(f"\n{name} Dataset")
    print("File exists:", path.exists())
    print("Shape:", saved_df.shape)
    print("Matches working dataset:",saved_df.equals(original_datasets[name]))


Integrated Dataset
File exists: True
Shape: (101525, 53)
Matches working dataset: True

Classification Dataset
File exists: True
Shape: (101525, 12)
Matches working dataset: True

Regression Dataset
File exists: True
Shape: (101525, 12)
Matches working dataset: True
